# MovieLens Recommendation System

## Overview
This notebook implements a recommendation system using the MovieLens dataset.
We will explore the data, build baseline models (Popularity), and implement Collaborative Filtering (SVD).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

## 1. Data Loading
Loading `ratings.csv` and `movies.csv`.

In [2]:
data_path = 'data/'

# Load Movies
movies = pd.read_csv(data_path + 'movies.csv')
print(f"Movies: {movies.shape}")
display(movies.head())

# Load Ratings (use chunks or sample if too large, but let's try full load first)
# Note: ratings.csv can be large (~900MB). If memory issues occur, we'll sample.
try:
    ratings = pd.read_csv(data_path + 'ratings.csv')
    print(f"Ratings: {ratings.shape}")
    display(ratings.head())
except Exception as e:
    print(f"Error loading ratings: {e}")

Movies: (86537, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Ratings: (33832162, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,1225734739
1,1,110,4.0,1225865086
2,1,158,4.0,1225733503
3,1,260,4.5,1225735204
4,1,356,5.0,1225735119


## 2. Exploratory Data Analysis (EDA)
Understanding the distribution of ratings and popularity of movies.

In [ ]:
# Ratings Distribution
plt.figure(figsize=(10, 5))
sns.countplot(x='rating', data=ratings, palette='viridis')
plt.title('Distribution of Ratings')
plt.show()

/tmp/ipykernel_1345/3611185548.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(x='rating', data=ratings, palette='viridis')


In [ ]:
# Most Rated Movies
rating_counts = ratings.groupby('movieId')['rating'].count().reset_index()
rating_counts.columns = ['movieId', 'rating_count']

top_movies = rating_counts.sort_values('rating_count', ascending=False).head(10)
top_movies = top_movies.merge(movies, on='movieId')

plt.figure(figsize=(12, 6))
sns.barplot(x='rating_count', y='title', data=top_movies, palette='magma')
plt.title('Top 10 Most Rated Movies')
plt.xlabel('Number of Ratings')
plt.show()

## 3. Baseline Model: Popularity Based
Recommend movies with the highest weighted rating (considering vote count).

In [ ]:
# Calculate Weighted Rating (IMDB formula)
# WR = (v / (v+m)) * R + (m / (v+m)) * C
# v = number of votes for the movie
# m = minimum votes required to be listed
# R = average rating of the movie
# C = mean vote across the whole report

C = ratings['rating'].mean()
m = rating_counts['rating_count'].quantile(0.90) # Top 10% most rated

qualified_movies = rating_counts.copy().loc[rating_counts['rating_count'] >= m]
qualified_movies = qualified_movies.merge(movies, on='movieId')

# Calculate Average Rating (R)
avg_ratings = ratings.groupby('movieId')['rating'].mean().reset_index()
avg_ratings.columns = ['movieId', 'avg_rating']
qualified_movies = qualified_movies.merge(avg_ratings, on='movieId')

def weighted_rating(x, m=m, C=C):
    v = x['rating_count']
    R = x['avg_rating']
    return (v/(v+m) * R) + (m/(v+m) * C)

qualified_movies['score'] = qualified_movies.apply(weighted_rating, axis=1)

# Top 10 Popular Movies
top_10_popular = qualified_movies.sort_values('score', ascending=False).head(10)
display(top_10_popular[['title', 'rating_count', 'avg_rating', 'score']])